In [ ]:
# Import libraries

import pandas as pd, sqlalchemy as sqla, decouple
import numpy as np
import decouple
from IPython.display import display
from datetime import date, datetime
import xlsxwriter
import re
import calendar
from dateutil.relativedelta import relativedelta

clone_engine = sqla.create_engine(decouple.config("MIS_DB"))
mis_fldr_path = decouple.config("MIS_DB")
mis_file_path = decouple.config("MIS_FILE_PATH")

pd.set_option('display.max_rows', None)      # Remove row limit
pd.set_option('display.max_columns', None)   # Remove column limit
pd.set_option('display.max_colwidth', None) 

In [154]:
# Import reference tables from Clone server database

# Account Detals
ref_account_details = pd.read_sql("SELECT * FROM ref.account_details", clone_engine)

# Store Details
ref_store_details = pd.read_sql("SELECT * FROM ref.store_details", clone_engine)

# Merchandiser Names
ref_merchandiser_names = pd.read_sql("SELECT * FROM ref.merchandiser_names", clone_engine)

# Store assignment 
ref_store_assignment = pd.read_sql("SELECT * FROM ref.store_assignment WHERE year = 2026", clone_engine)

# TOS Names 
ref_tos_names = pd.read_sql("SELECT * FROM ref.tos_names", clone_engine)

# Store Sales
store_sales = pd.read_sql(r"""
        -- OUTRIGHT
        SELECT
            sin.`date`, ref_sd.bpc_store_code, SUM(net_amount) AS net_amount
        FROM sales.or_qb AS sin

        LEFT JOIN ref.account_names AS ref_an
            ON sin.`name` = ref_an.`name`

        LEFT JOIN ref.account_details AS ref_ad
            ON ref_an.account_name = ref_ad.account_name

        LEFT JOIN ref.store_codes AS ref_sc
            ON ref_an.account_name = ref_sc.account_name
            AND COALESCE(sin.ship_to_address_1, '') = COALESCE(ref_sc.ship_to_address_1, '')
            AND COALESCE(sin.ship_to_address_2, '') = COALESCE(ref_sc.ship_to_address_2, '')

        LEFT JOIN ref.store_details AS ref_sd
            ON ref_sc.account_name = ref_sd.account_name
            AND ref_sc.store_code = ref_sd.store_code

        WHERE YEAR(sin.`date`) > 2024
            AND ref_ad.account_chain NOT IN (
                'BPC Employee',
                'DMD Account',
                'DSAP'
            )
            AND ref_an.account_name NOT IN (
            -- with Sell-Out Accounts
                'Watsons Personal Care Store (Phils.) Inc.',
                'Philippine Seven Corporation',
                'South Star Drug',
                'Alfamart Trading Philippines, Inc.',
                'Robinsons Supermarket Corporation',
                'Rose Pharmacy, Incorporated',
            -- allocation based accounts
                'Mercury Drug Corporation',
                'Puregold Price Club Inc.',
                'Pasig Supermarket Inc.',
                'Sanford Marketing Corporation',
                'Supervalue Inc.',
                'Super Shopping Market, Inc.',
            -- SOB based accounts
                'Alfamart Trading Philippines, Inc.',
                'Bohol Quality Corporation',
                'Cecile\'s Pharmacy & Convenience Store',
                'Greatshop Supermarket',
                'JT & AC Beauty Products Trading',
                'KJ Fairmart, Inc.',
                'Pricewise Marketing Corporation',
                'Prince Retail Merchandising Inc.',
                'Rika Drugstore & Distribution Corp.',
                'St. Joseph Drug Store',
                'ThreeSixty Pharmacy'
            ) 
        GROUP BY
            sin.`date`, ref_sd.bpc_store_code

        UNION ALL

        -- CONSIGNMENT FROM QB
        SELECT
            STR_TO_DATE(CONCAT(YEAR(cn.`date`),'-', `month`,'-01'), '%Y-%M-%d ') AS `date`,
            ref_sd.bpc_store_code,
            SUM(net_amount) AS net_amount 
        FROM sales.cn_qb AS cn

        LEFT JOIN ref.account_names AS ref_an
            ON cn.`name` = ref_an.`name`

        LEFT JOIN ref.account_details AS ref_ad
            ON ref_an.account_name = ref_ad.account_name

        LEFT JOIN ref.store_codes AS ref_sc
            ON ref_an.account_name = ref_sc.account_name
            AND COALESCE(cn.ship_to_address_1, '') = COALESCE(ref_sc.ship_to_address_1, '')
            AND COALESCE(cn.ship_to_address_2, '') = COALESCE(ref_sc.ship_to_address_2, '')

        LEFT JOIN ref.store_details AS ref_sd
            ON ref_sc.account_name = ref_sd.account_name
            AND ref_sc.store_code = ref_sd.store_code

        WHERE YEAR(cn.`date`) > 2024
        GROUP BY
            STR_TO_DATE(CONCAT(YEAR(cn.`date`),'-', `month`,'-01'), '%Y-%M-%d '), ref_sd.bpc_store_code

        UNION ALL

        -- SELL-OUT AND CONSIGNMENT FROM PORTAL
        SELECT
            STR_TO_DATE(CONCAT(sout.`year`,'-', sout.`month`,'-01'), '%Y-%M-%d ') AS `date`,
            ref_sd.bpc_store_code,
            SUM(sout.net_amount) AS net_amount
        FROM (
            SELECT 
            `year`, `month`, account_name, store_code, item_code, SUM(qty) AS qty, SUM(amount) AS amount, SUM(net_amount) AS net_amount
            FROM sales.sout_prtl
            WHERE `year` > 2024
            GROUP BY `year`, `month`, account_name, store_code, item_code
            
            UNION ALL
            
            SELECT `year`, `month`, account_name, store_code, item_code, SUM(qty) AS qty, SUM(amount) AS amount, SUM(net_amount) AS net_amount
            FROM sales.cn_prtl
            WHERE `year` > 2024
            GROUP BY `year`, `month`, account_name, store_code, item_code
            
        ) sout

        -- Add account details
        LEFT JOIN ref.account_details AS ref_ad
            ON sout.account_name = ref_ad.account_name

        -- Add store details
        LEFT JOIN ref.store_details AS ref_sd
            ON sout.account_name = ref_sd.account_name
            AND sout.store_code = ref_sd.store_code

        GROUP BY `date`, ref_sd.bpc_store_code

        UNION ALL

        SELECT
            STR_TO_DATE(CONCAT(sob.`year`,'-', sob.`month`,'-01'), '%Y-%M-%d ') AS `date`,
            ref_sd.bpc_store_code,
            sob.count * sub.net_amount AS net_amount
        FROM (
            SELECT * FROM ref.store_count WHERE `year` > 2024
        ) sob

        LEFT JOIN (
            SELECT
                sub.`year`, sub.`month`,
                sub.account_name,
                sub.net_amount / ref_sc.count AS net_amount 
            FROM (
                SELECT
                    YEAR(sin.`date`) AS `year`,
                    MONTHNAME(sin.`date`) AS `month`,
                    ref_an.account_name,
                    SUM(net_amount) AS net_amount
                FROM sales.or_qb AS sin

                LEFT JOIN ref.account_names AS ref_an
                    ON sin.`name` = ref_an.`name`

                WHERE YEAR(sin.`date`) > 2024
                    AND ref_an.account_name IN (
                    -- SOB based accounts
                        'Alfamart Trading Philippines, Inc.',
                        'Bohol Quality Corporation',
                        'Cecile\'s Pharmacy & Convenience Store',
                        'Greatshop Supermarket',
                        'JT & AC Beauty Products Trading',
                        'KJ Fairmart, Inc.',
                        'Pricewise Marketing Corporation',
                        'Prince Retail Merchandising Inc.',
                        'Rika Drugstore & Distribution Corp.',
                        'St. Joseph Drug Store',
                        'ThreeSixty Pharmacy'
                    )
                GROUP BY `year`, `month`, ref_an.account_name
            ) sub

            LEFT JOIN (
                SELECT `year`, `month`, account_name, SUM(`count`) AS `count`
                FROM ref.store_count
                WHERE `year` > 2024
                GROUP BY `year`, `month`, account_name
            ) ref_sc
                ON sub.`year` = ref_sc.`year`
                AND sub.`month` = ref_sc.`month`
                AND sub.account_name = ref_sc.account_name
        ) sub
            ON sob.`year` = sub.`year`
            AND sob.`month` = sub.`month`
            AND sob.account_name = sub.account_name

        LEFT JOIN ref.store_details AS ref_sd
            ON sob.account_name = ref_sd.account_name
            AND sob.store_code = ref_sd.store_code;
""", clone_engine)

display(ref_account_details.head())
display(ref_store_details.head())
display(ref_store_assignment.head())
display(ref_merchandiser_names.head())
display(store_sales.head())

,account_name,account_chain,bpc_region,account_channel,channel_class,business_unit,account_type,sales_group,report_group,order,lead_id
0,456 Supermarket Inc.,456 Supermarket Inc.,North Luzon,Supermarket,Local Account,Regional Sales,Outright,S4,North Luzon,11,REG-001
1,AANJ Prosperity and Wealth Mdse Corp.,AANJ Prosperity and Wealth Mdse Corp.,NCR,Convenience Store,Local Account,Regional Sales,Outright,S4,NCR,12,REG-002
2,"AB Leisure Exponent, Inc.","AB Leisure Exponent, Inc.",NCR,Others,Local Account,Regional Sales,Outright,S4,NCR,12,REG-002
3,Abecure Wellness Products Trading,Abecure Wellness Products Trading,Visayas,Distributor,Local Account,Regional Sales,Outright,S4,Visayas,14,REG-004
4,ACC Hypermart Corporation,ACC Hypermart Corporation,South Luzon,Supermarket,Local Account,Regional Sales,Outright,S4,South Luzon,13,REG-003


,account_name,store_name,store_code,bpc_store_code,city,province,region,bpc_region,remarks
0,AANJ Prosperity and Wealth Mdse Corp.,Aanj Prosperity And Wealth Mdse Corp.,AANJ01,BPC-APW-00001,Valenzuela,Metro Manila,NCR,NCR,None
1,"AB Leisure Exponent, Inc.","Ab Leisure Exponent, Inc.",ABLE01,BPC-ALE-00001,Mandaluyong,Metro Manila,NCR,NCR,None
2,Abecure Wellness Products Trading,Abecure Wellness Products Trading,AWPT01,BPC-AWP-00001,Iloilo City,Iloilo,Region VI,Visayas,None
3,ACC Hypermart Corporation,Acc Hypermart Corporation Virac Sunrise,ACC01,BPC-AHC-00001,Virac,Catanduanes,Region V,South Luzon,None
4,ACC Hypermart Corporation,Acc Hypermart Corporation,ACC02,BPC-AHC-00002,Virac,Catanduanes,Region V,South Luzon,None


,account_name,store_code,year,month,plantilla_code
0,AANJ Prosperity and Wealth Mdse Corp.,AANJ01,2026,January,-
1,AANJ Prosperity and Wealth Mdse Corp.,AANJ01,2026,February,-
2,AANJ Prosperity and Wealth Mdse Corp.,AANJ01,2026,March,-
3,AANJ Prosperity and Wealth Mdse Corp.,AANJ01,2026,April,-
4,"AB Leisure Exponent, Inc.",ABLE01,2026,January,-


,plantilla_code,year,month,merchandiser
0,CAR-BA-01,2026,January,Vacant
1,CAR-BA-01,2026,February,Vacant
2,CAR-BA-01,2026,March,Vacant
3,CAR-BA-01,2026,April,Vacant
4,CAR-BA-02,2026,January,Vacant


,date,bpc_store_code,net_amount
0,2025-01-08,BPC-DWM-00001,12290.39280
1,2025-01-10,BPC-BAD-00001,2745.89280
2,2025-01-13,BPC-CVA-00001,3067.20000
3,2025-01-13,BPC-CVA-00002,3067.20000
4,2025-01-14,BPC-DCC-00137,1439.99744


In [155]:
# Import data from created Dataframe

# Sales Basis
raw_sales_basis = {
    "store_name": [
        'Mercury Drug Corporation',
        'Puregold',
        'Cash & Carry Department Store',
        'Ever Supermarket',
        'Gaisano Grand',
        'Metro Gaisano',
        'Philippine Seven Corporation',
        'Robinsons Supermarket Corporation',
        'Rose Pharmacy, Incorporated',
        'South Star Drug',
        'Sta. Lucia East Department Store, Inc.',
        'Sta. Lucia East Supermarket',
        'Synagie Inc.',
        'Tropical Hut Super Market',
        'UM Superfoods Corp.',
        'Unimart, Incorporated',
        'Waltermart Supermarket Inc.',
        'Watsons Personal Care Store (Phils.) Inc.'
    ],
    "sales_basis": [
        "Allocation", 
        "Allocation", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out", 
        "Sell Out"
        ]
}

sales_basis = pd.DataFrame(raw_sales_basis, columns=["store_name", "sales_basis"])
sales_basis

,store_name,sales_basis
0,Mercury Drug Corporation,Allocation
1,Puregold,Allocation
2,Cash & Carry Department Store,Sell Out
3,Ever Supermarket,Sell Out
4,Gaisano Grand,Sell Out
5,Metro Gaisano,Sell Out
6,Philippine Seven Corporation,Sell Out
7,Robinsons Supermarket Corporation,Sell Out
8,"Rose Pharmacy, Incorporated",Sell Out
9,South Star Drug,Sell Out


In [ ]:
# Import 2026 targets from Excel file

targets_2026 = pd.read_excel(rf"{mis_file_path}MIS Data\Universe\Python Generated Universe\Universe source files\2026 Target - Universe x Target.xlsx")
targets_2026["key"] = targets_2026["account_name"] + targets_2026["store_code"].astype(str).str.lstrip("0")
targets_2026 = targets_2026.drop(columns="bpc_store_code")

# Prepare the bpc_store_codes to be attached to the targets table
bpc_store_code_list = ref_store_details.copy()
bpc_store_code_list["store_code"] = bpc_store_code_list["store_code"].str.lstrip("0")
bpc_store_code_list["key"] = bpc_store_code_list["account_name"] + bpc_store_code_list["store_code"]
bpc_store_code_list = bpc_store_code_list[["key", "bpc_store_code"]]

targets_2026 = pd.merge(targets_2026, bpc_store_code_list, how="left", on="key")

targets_2026 = targets_2026[[
    #    'account_name', 'store_code', key', 
       'bpc_store_code',
       'January', 'February', 'March',
       'April', 'May', 'June', 
       'July', 'August', 'September', 
       'October', 'November', 'December'
       ]]

targets_2026 = targets_2026.rename(columns={
    "January": "January Target",
    "February": "February Target",
    "March": "March Target",
    "April": "April Target",
    "May": "May Target",
    "June": "June Target",
    "July": "July Target",
    "August": "August Target",
    "September": "September Target",
    "October": "October Target",
    "November": "November Target",
    "December": "December Target"
})

# # non_zero_targets = targets_2026[targets_2026["January Target"] != 0]
null_bpc_codes = targets_2026[targets_2026["bpc_store_code"].isna()]
print("null_bpc_codes.shape: ", null_bpc_codes.shape)
display(null_bpc_codes)

# print(bpc_store_code_list.columns)
# print(targets_2026.columns)

with pd.ExcelWriter(r"C:\eli_dump\Backup\MISC\targets_2026_bpc_checker.xlsx", engine="xlsxwriter") as writer:
    bpc_store_code_list.to_excel(writer, sheet_name="bpc_store_codes", index=False)
    targets_2026.to_excel(writer, sheet_name="targets_2026_S1_S3", index=False)

null_bpc_codes.shape:  (0, 13)


,bpc_store_code,January Target,February Target,March Target,April Target,May Target,June Target,July Target,August Target,September Target,October Target,November Target,December Target


In [ ]:
# Import 2026 S4 targets from Excel file

# bpc_store_code_list

S4_targets_2026 = pd.read_excel(rf"{mis_file_path}MIS Data\Universe\Python Generated Universe\Universe source files\S4 Target.xlsx")
S4_targets_2026["key"] = S4_targets_2026["Account"] + S4_targets_2026["Store Code"].astype(str).str.lstrip("0")
S4_targets_2026 = S4_targets_2026.rename(columns={"Account": "account_name", "Store Code": "store_code"})

S4_targets_2026 = pd.merge(S4_targets_2026, bpc_store_code_list, how="left", on="key")
print(S4_targets_2026.columns)

S4_targets_2026 = S4_targets_2026[[
    #    'account_name', 'store_code', 'key',
       'bpc_store_code',
       'January', 'February', 'March', 
       'April', 'May', 'June', 
       'July', 'August', 'September', 
       'October', 'November', 'December'
       ]]

S4_targets_2026 = S4_targets_2026.rename(columns={
    "January": "January S4 Target",
    "February": "February S4 Target",
    "March": "March S4 Target",
    "April": "April S4 Target",
    "May": "May S4 Target",
    "June": "June S4 Target",
    "July": "July S4 Target",
    "August": "August S4 Target",
    "September": "September S4 Target",
    "October": "October S4 Target",
    "November": "November S4 Target",
    "December": "December S4 Target"
})

null_s4_bpc_codes = S4_targets_2026[S4_targets_2026["bpc_store_code"].isna()]
display("null_s4_bpc_codes: ", null_s4_bpc_codes)

Index(['Area', 'account_name', 'store_code', 'Key', 'Store Name', 'Region',
       '2026 Avg Sales', 'Contribution %', 'Annual Target', 'January',
       'February', 'March', 'April', 'May', 'June', 'July', 'August',
       'September', 'October', 'November', 'December', 'Note', 'key',
       'bpc_store_code'],
      dtype='object')


'null_s4_bpc_codes: '

,bpc_store_code,January S4 Target,February S4 Target,March S4 Target,April S4 Target,May S4 Target,June S4 Target,July S4 Target,August S4 Target,September S4 Target,October S4 Target,November S4 Target,December S4 Target


In [ ]:
# Import RED Approved Schedule from Excel file

red_approved_schedule = pd.read_excel(rf"{mis_file_path}MIS Data\RED Alert\RED Bulk Upload.xlsx", sheet_name="Approved Schedule")
red_approved_schedule.head()

,Employee Code,Name,TOS,Account Name,Store Name,Store Code,Week,Day,City,Province,Region,BPC Region
0,RIII-MPA-01,"Espino, Denn Mark","Illustre, Angelita",Mercury Drug Corporation,Mercury Drug Corporation Bulacan Balagtas Town Center,0481,1,Monday,Balagtas,Bulacan,Region III,North Luzon
1,RIII-MPA-01,"Espino, Denn Mark","Illustre, Angelita",Mercury Drug Corporation,Mercury Drug Corporation Bulacan Balagtas Town Center,0481,1,Friday,Balagtas,Bulacan,Region III,North Luzon
2,RIII-MPA-01,"Espino, Denn Mark","Illustre, Angelita",Watsons Personal Care Store (Phils.) Inc.,Watsons Personal Care Store (Phils.) Inc. Balagtas Town Center,1318,1,Monday,Balagtas,Bulacan,Region III,North Luzon
3,RIII-MPA-01,"Espino, Denn Mark","Illustre, Angelita",Watsons Personal Care Store (Phils.) Inc.,Watsons Personal Care Store (Phils.) Inc. Balagtas Town Center,1318,1,Friday,Balagtas,Bulacan,Region III,North Luzon
4,RIII-MPA-01,"Espino, Denn Mark","Illustre, Angelita",Robinsons Supermarket Corporation,Robinsons Supermarket Corporation Rs Balagtas Towncenter,00213,1,Monday,Balagtas,Bulacan,Region III,North Luzon


In [159]:
# Join account and store details

account_store_details_merge = pd.merge(ref_store_details, ref_account_details, how="left", on="account_name")

account_store_details_merge = account_store_details_merge[["channel_class", "account_type", "account_channel", "account_name", "store_name", "store_code", "city", "province", "region", "bpc_region_y", "sales_group", "bpc_store_code"]]
account_store_details_merge = account_store_details_merge.rename(columns={"bpc_region_y": "bpc_region"})

print(account_store_details_merge.columns)
print(f"\nDataframe dimensions: \naccount_details: {ref_account_details.shape} \nstore_details: {ref_store_details.shape}")
display(f"Merged Table Shape: {account_store_details_merge.shape}")
display(account_store_details_merge.head())

Index(['channel_class', 'account_type', 'account_channel', 'account_name',
       'store_name', 'store_code', 'city', 'province', 'region', 'bpc_region',
       'sales_group', 'bpc_store_code'],
      dtype='object')

Dataframe dimensions: 
account_details: (2164, 11) 
store_details: (13747, 9)


'Merged Table Shape: (13747, 12)'

,channel_class,account_type,account_channel,account_name,store_name,store_code,city,province,region,bpc_region,sales_group,bpc_store_code
0,Local Account,Outright,Convenience Store,AANJ Prosperity and Wealth Mdse Corp.,Aanj Prosperity And Wealth Mdse Corp.,AANJ01,Valenzuela,Metro Manila,NCR,NCR,S4,BPC-APW-00001
1,Local Account,Outright,Others,"AB Leisure Exponent, Inc.","Ab Leisure Exponent, Inc.",ABLE01,Mandaluyong,Metro Manila,NCR,NCR,S4,BPC-ALE-00001
2,Local Account,Outright,Distributor,Abecure Wellness Products Trading,Abecure Wellness Products Trading,AWPT01,Iloilo City,Iloilo,Region VI,Visayas,S4,BPC-AWP-00001
3,Local Account,Outright,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation Virac Sunrise,ACC01,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00001
4,Local Account,Outright,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation,ACC02,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00002


In [160]:
# Join with sales basis

sales_basis_merge = pd.merge(account_store_details_merge, sales_basis, how="left", on="store_name")
sales_basis_merge["sales_basis"] = sales_basis_merge["sales_basis"].fillna("Sell In")

non_null = sales_basis_merge[~sales_basis_merge["sales_basis"].isna()]
non_null.head()

sales_basis_merge = sales_basis_merge[[
    'channel_class', 'account_type', 'sales_basis', 'account_channel', 'account_name',
    'store_name', 'store_code', 'city', 'province', 'region', 'bpc_region', 'sales_group', 'bpc_store_code'
]]

sales_basis_merge["key"] = sales_basis_merge["account_name"] + sales_basis_merge["store_code"]
sales_basis_merge.head()

,channel_class,account_type,sales_basis,account_channel,account_name,store_name,store_code,city,province,region,bpc_region,sales_group,bpc_store_code,key
0,Local Account,Outright,Sell In,Convenience Store,AANJ Prosperity and Wealth Mdse Corp.,Aanj Prosperity And Wealth Mdse Corp.,AANJ01,Valenzuela,Metro Manila,NCR,NCR,S4,BPC-APW-00001,AANJ Prosperity and Wealth Mdse Corp.AANJ01
1,Local Account,Outright,Sell In,Others,"AB Leisure Exponent, Inc.","Ab Leisure Exponent, Inc.",ABLE01,Mandaluyong,Metro Manila,NCR,NCR,S4,BPC-ALE-00001,"AB Leisure Exponent, Inc.ABLE01"
2,Local Account,Outright,Sell In,Distributor,Abecure Wellness Products Trading,Abecure Wellness Products Trading,AWPT01,Iloilo City,Iloilo,Region VI,Visayas,S4,BPC-AWP-00001,Abecure Wellness Products TradingAWPT01
3,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation Virac Sunrise,ACC01,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00001,ACC Hypermart CorporationACC01
4,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation,ACC02,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00002,ACC Hypermart CorporationACC02


In [161]:
# Add Plantilla Code

# Join store_assignment, merchandiser_names, and tos_names
filtered_store_assignment = ref_store_assignment[ref_store_assignment["month"] == "July"]
filtered_store_assignment = filtered_store_assignment.drop(columns={"year", "month"})
filtered_store_assignment = filtered_store_assignment.drop_duplicates()
# display(ref_store_assignment.head())

store_assignment_merge = pd.merge(sales_basis_merge, filtered_store_assignment, how="left",  on=["account_name", "store_code"])

print(store_assignment_merge.columns)
display(store_assignment_merge.head())

Index(['channel_class', 'account_type', 'sales_basis', 'account_channel',
       'account_name', 'store_name', 'store_code', 'city', 'province',
       'region', 'bpc_region', 'sales_group', 'bpc_store_code', 'key',
       'plantilla_code'],
      dtype='object')


,channel_class,account_type,sales_basis,account_channel,account_name,store_name,store_code,city,province,region,bpc_region,sales_group,bpc_store_code,key,plantilla_code
0,Local Account,Outright,Sell In,Convenience Store,AANJ Prosperity and Wealth Mdse Corp.,Aanj Prosperity And Wealth Mdse Corp.,AANJ01,Valenzuela,Metro Manila,NCR,NCR,S4,BPC-APW-00001,AANJ Prosperity and Wealth Mdse Corp.AANJ01,RIII-MPA-01
1,Local Account,Outright,Sell In,Others,"AB Leisure Exponent, Inc.","Ab Leisure Exponent, Inc.",ABLE01,Mandaluyong,Metro Manila,NCR,NCR,S4,BPC-ALE-00001,"AB Leisure Exponent, Inc.ABLE01",None
2,Local Account,Outright,Sell In,Distributor,Abecure Wellness Products Trading,Abecure Wellness Products Trading,AWPT01,Iloilo City,Iloilo,Region VI,Visayas,S4,BPC-AWP-00001,Abecure Wellness Products TradingAWPT01,None
3,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation Virac Sunrise,ACC01,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00001,ACC Hypermart CorporationACC01,None
4,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation,ACC02,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00002,ACC Hypermart CorporationACC02,None


In [162]:
# Merchandiser Name

filtered_merchandiser_names = ref_merchandiser_names[ref_merchandiser_names["month"] == "July"]
filtered_merchandiser_names = filtered_merchandiser_names.drop(columns={"year", "month"})
filtered_merchandiser_names = filtered_merchandiser_names.drop_duplicates()
# display(ref_merchandiser_names.head())

merchandiser_merge = pd.merge(store_assignment_merge, filtered_merchandiser_names, how="left", on="plantilla_code")
not_vacant_disers = merchandiser_merge[
    merchandiser_merge["plantilla_code"].notna() & (merchandiser_merge["plantilla_code"] != "-")
]

diser_coding_regex = r"\D+-(\D+)-\d+"

merchandiser_merge["diser_coding"] = merchandiser_merge["plantilla_code"].str.extract(diser_coding_regex)
display(merchandiser_merge.head())
# display(not_vacant_disers.head())

,channel_class,account_type,sales_basis,account_channel,account_name,store_name,store_code,city,province,region,bpc_region,sales_group,bpc_store_code,key,plantilla_code,merchandiser,diser_coding
0,Local Account,Outright,Sell In,Convenience Store,AANJ Prosperity and Wealth Mdse Corp.,Aanj Prosperity And Wealth Mdse Corp.,AANJ01,Valenzuela,Metro Manila,NCR,NCR,S4,BPC-APW-00001,AANJ Prosperity and Wealth Mdse Corp.AANJ01,RIII-MPA-01,"Espino, Denn Mark",MPA
1,Local Account,Outright,Sell In,Others,"AB Leisure Exponent, Inc.","Ab Leisure Exponent, Inc.",ABLE01,Mandaluyong,Metro Manila,NCR,NCR,S4,BPC-ALE-00001,"AB Leisure Exponent, Inc.ABLE01",None,NaN,NaN
2,Local Account,Outright,Sell In,Distributor,Abecure Wellness Products Trading,Abecure Wellness Products Trading,AWPT01,Iloilo City,Iloilo,Region VI,Visayas,S4,BPC-AWP-00001,Abecure Wellness Products TradingAWPT01,None,NaN,NaN
3,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation Virac Sunrise,ACC01,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00001,ACC Hypermart CorporationACC01,None,NaN,NaN
4,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation,ACC02,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00002,ACC Hypermart CorporationACC02,None,NaN,NaN


In [163]:
# Tos Names

filtered_tos_names = ref_tos_names[ref_tos_names["month"] == "July"]
filtered_tos_names = filtered_tos_names.drop(columns={"year","month"})

# ref_tos_names

tos_names_merge = pd.merge(merchandiser_merge, filtered_tos_names, how="left", on=["account_name", "store_code"])
tos_names_merge.head()

,channel_class,account_type,sales_basis,account_channel,account_name,store_name,store_code,city,province,region,bpc_region,sales_group,bpc_store_code,key,plantilla_code,merchandiser,diser_coding,tos
0,Local Account,Outright,Sell In,Convenience Store,AANJ Prosperity and Wealth Mdse Corp.,Aanj Prosperity And Wealth Mdse Corp.,AANJ01,Valenzuela,Metro Manila,NCR,NCR,S4,BPC-APW-00001,AANJ Prosperity and Wealth Mdse Corp.AANJ01,RIII-MPA-01,"Espino, Denn Mark",MPA,Angelita Illustre
1,Local Account,Outright,Sell In,Others,"AB Leisure Exponent, Inc.","Ab Leisure Exponent, Inc.",ABLE01,Mandaluyong,Metro Manila,NCR,NCR,S4,BPC-ALE-00001,"AB Leisure Exponent, Inc.ABLE01",None,NaN,NaN,None
2,Local Account,Outright,Sell In,Distributor,Abecure Wellness Products Trading,Abecure Wellness Products Trading,AWPT01,Iloilo City,Iloilo,Region VI,Visayas,S4,BPC-AWP-00001,Abecure Wellness Products TradingAWPT01,None,NaN,NaN,None
3,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation Virac Sunrise,ACC01,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00001,ACC Hypermart CorporationACC01,None,NaN,NaN,None
4,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation,ACC02,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00002,ACC Hypermart CorporationACC02,None,NaN,NaN,None


In [164]:
# Join with Sales Data

store_sales_copy = store_sales.copy()
# filtered_store_sales.head()

# print(filtered_store_sales["date"].apply(type))
store_sales_copy["date"] = pd.to_datetime(store_sales_copy["date"])
store_sales_copy["month"] = store_sales_copy["date"].dt.month_name()
store_sales_copy["year"] = store_sales_copy["date"].dt.year

store_sales_copy = store_sales_copy.drop(columns=["date"])
store_sales_copy = store_sales_copy[["year", "month", "bpc_store_code", "net_amount"]]

store_sales_2025 = store_sales_copy[store_sales_copy["year"] == 2025]
store_sales_2025 = store_sales_2025.drop(columns="month")
store_sales_2025 = store_sales_2025.groupby(["year", "bpc_store_code"]).agg({"net_amount": "sum"}).reset_index()
store_sales_2025["net_amount"] = store_sales_2025["net_amount"] / 12

store_sales_2026 = store_sales_copy[store_sales_copy["year"] == 2026]


store_sales_2025 = store_sales_2025.rename(columns={"net_amount":"2025 Average Sales"})

display(store_sales_2025.head())
display(store_sales_2026.head())
# display(store_sales_copy.head())

# store_sales_copy = store_sales_copy.drop(["date"])
# store_sales_copy = store_sales_copy.groupby()

,year,bpc_store_code,2025 Average Sales
0,2025,BPC-ABS-00001,5633.100480
1,2025,BPC-ACT-00001,26635.050480
2,2025,BPC-ADD-00001,5886.999253
3,2025,BPC-AHC-00001,1169.551320
4,2025,BPC-AHC-00002,1567.799520


,year,month,bpc_store_code,net_amount
844,2026,January,BPC-DSG-00001,12740.24640
845,2026,January,BPC-ESI-00001,25272.00144
846,2026,January,BPC-ADD-00001,21182.39872
847,2026,January,BPC-DCC-00139,1439.99744
848,2026,January,BPC-CTI-00001,25090.80672


In [165]:
# Prepare the 2026 Sales Dataframe

# Create month_name columns from the "month" column
month_order = list(calendar.month_name)[1:] 

# convert existing month column to ordered categorical, in place
store_sales_2026["month"] = pd.Categorical(
    store_sales_2026["month"], categories=month_order, ordered=True
)

store_sales_2026 = store_sales_2026.pivot_table(
    index="bpc_store_code",       
    columns="month",
    values="net_amount",
    aggfunc="sum",
    fill_value=0
)

store_sales_2026 = store_sales_2026.rename_axis(None, axis=1).reset_index()

display(store_sales_2026.columns)
store_sales_2026.head()

C:\Users\bioco\AppData\Local\Temp\ipykernel_21760\2548885720.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  store_sales_2026["month"] = pd.Categorical(
C:\Users\bioco\AppData\Local\Temp\ipykernel_21760\2548885720.py:11: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  store_sales_2026 = store_sales_2026.pivot_table(


Index(['bpc_store_code', 'January', 'February', 'March', 'April', 'May',
       'June', 'July', 'August', 'September', 'October', 'November',
       'December'],
      dtype='object')

,bpc_store_code,January,February,March,April,May,June,July,August,September,October,November,December
0,BPC-ACT-00001,70361.99856,0.0,0.00000,0.0,75038.40288,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,BPC-ADD-00001,21182.39872,0.0,29913.58720,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,BPC-AHC-00001,0.00000,0.0,0.00000,0.0,5923.79424,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,BPC-AHC-00003,0.00000,0.0,0.00000,0.0,3509.99712,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,BPC-APW-00001,0.00000,0.0,8629.18504,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [166]:
# Details and combined Sales Data

print("tos_names_merge: ", tos_names_merge.columns, "\n")
print("store_sales: ", store_sales_2026.columns,  "\n")

sales_per_account_merge = pd.merge(tos_names_merge, store_sales_2026, how="left", on="bpc_store_code")

display(sales_per_account_merge.head())

tos_names_merge:  Index(['channel_class', 'account_type', 'sales_basis', 'account_channel',
       'account_name', 'store_name', 'store_code', 'city', 'province',
       'region', 'bpc_region', 'sales_group', 'bpc_store_code', 'key',
       'plantilla_code', 'merchandiser', 'diser_coding', 'tos'],
      dtype='object') 

store_sales:  Index(['bpc_store_code', 'January', 'February', 'March', 'April', 'May',
       'June', 'July', 'August', 'September', 'October', 'November',
       'December'],
      dtype='object') 



,channel_class,account_type,sales_basis,account_channel,account_name,store_name,store_code,city,province,region,bpc_region,sales_group,bpc_store_code,key,plantilla_code,merchandiser,diser_coding,tos,January,February,March,April,May,June,July,August,September,October,November,December
0,Local Account,Outright,Sell In,Convenience Store,AANJ Prosperity and Wealth Mdse Corp.,Aanj Prosperity And Wealth Mdse Corp.,AANJ01,Valenzuela,Metro Manila,NCR,NCR,S4,BPC-APW-00001,AANJ Prosperity and Wealth Mdse Corp.AANJ01,RIII-MPA-01,"Espino, Denn Mark",MPA,Angelita Illustre,0.0,0.0,8629.18504,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Local Account,Outright,Sell In,Others,"AB Leisure Exponent, Inc.","Ab Leisure Exponent, Inc.",ABLE01,Mandaluyong,Metro Manila,NCR,NCR,S4,BPC-ALE-00001,"AB Leisure Exponent, Inc.ABLE01",None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Local Account,Outright,Sell In,Distributor,Abecure Wellness Products Trading,Abecure Wellness Products Trading,AWPT01,Iloilo City,Iloilo,Region VI,Visayas,S4,BPC-AWP-00001,Abecure Wellness Products TradingAWPT01,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation Virac Sunrise,ACC01,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00001,ACC Hypermart CorporationACC01,None,NaN,NaN,None,0.0,0.0,0.00000,0.0,5923.79424,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation,ACC02,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00002,ACC Hypermart CorporationACC02,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [167]:
# Prepare the 2025 Average Sales Dataframe

store_sales_2025 = store_sales_2025.drop(columns="year")

merged_2025_2026_sales = pd.merge(sales_per_account_merge, store_sales_2025, how="left", on="bpc_store_code")
merged_2025_2026_sales = merged_2025_2026_sales[[
       'channel_class', 'account_type', 'sales_basis', 'account_channel',
       'account_name', 'store_name', 'store_code', 'city', 'province',
       'region', 'bpc_region', 'sales_group', 'bpc_store_code', 'key',
       'plantilla_code', 'merchandiser', 'diser_coding', 'tos', '2025 Average Sales', 'January',
       'February', 'March', 'April', 'May', 'June', 'July', 'August',
       'September', 'October', 'November', 'December'
       ]]

display("2025 Average Sales table: ", store_sales_2025.head())
print("Merged 2025 Sales with main dataframe shape: ", merged_2025_2026_sales.shape)
display("Merged 2025 Sales with main dataframe: ", merged_2025_2026_sales.head())

'2025 Average Sales table: '

,bpc_store_code,2025 Average Sales
0,BPC-ABS-00001,5633.100480
1,BPC-ACT-00001,26635.050480
2,BPC-ADD-00001,5886.999253
3,BPC-AHC-00001,1169.551320
4,BPC-AHC-00002,1567.799520


Merged 2025 Sales with main dataframe shape:  (13747, 31)


'Merged 2025 Sales with main dataframe: '

,channel_class,account_type,sales_basis,account_channel,account_name,store_name,store_code,city,province,region,bpc_region,sales_group,bpc_store_code,key,plantilla_code,merchandiser,diser_coding,tos,2025 Average Sales,January,February,March,April,May,June,July,August,September,October,November,December
0,Local Account,Outright,Sell In,Convenience Store,AANJ Prosperity and Wealth Mdse Corp.,Aanj Prosperity And Wealth Mdse Corp.,AANJ01,Valenzuela,Metro Manila,NCR,NCR,S4,BPC-APW-00001,AANJ Prosperity and Wealth Mdse Corp.AANJ01,RIII-MPA-01,"Espino, Denn Mark",MPA,Angelita Illustre,NaN,0.0,0.0,8629.18504,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Local Account,Outright,Sell In,Others,"AB Leisure Exponent, Inc.","Ab Leisure Exponent, Inc.",ABLE01,Mandaluyong,Metro Manila,NCR,NCR,S4,BPC-ALE-00001,"AB Leisure Exponent, Inc.ABLE01",None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Local Account,Outright,Sell In,Distributor,Abecure Wellness Products Trading,Abecure Wellness Products Trading,AWPT01,Iloilo City,Iloilo,Region VI,Visayas,S4,BPC-AWP-00001,Abecure Wellness Products TradingAWPT01,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation Virac Sunrise,ACC01,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00001,ACC Hypermart CorporationACC01,None,NaN,NaN,None,1169.55132,0.0,0.0,0.00000,0.0,5923.79424,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation,ACC02,Virac,Catanduanes,Region V,South Luzon,S4,BPC-AHC-00002,ACC Hypermart CorporationACC02,None,NaN,NaN,None,1567.79952,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [168]:
# Join the Sales Table with the 2026 S1-S3 Targets table

sales_target_merge = pd.merge(merged_2025_2026_sales, targets_2026, how="left", on="bpc_store_code")

_y_columns = [column for column in sales_target_merge.columns if "_y" in column]

print(sales_target_merge.columns)

sales_target_merge = sales_target_merge[[    
    "channel_class", "account_type", "sales_basis",
    "account_channel","account_name","store_name","store_code","city","province","region",
    "bpc_region", "key", "plantilla_code",
    "diser_coding", "merchandiser", "tos", 
    "sales_group", "bpc_store_code", "2025 Average Sales",
    "January", "January Target",
    "February", "February Target",
    "March", "March Target",
    "April", "April Target",
    "May", "May Target",
    "June", "June Target",
    "July", "July Target",
    "August", "August Target",
    "September", "September Target",
    "October", "October Target",
    "November", "November Target",
    "December", "December Target",
    ]]

sales_target_merge.head()

Index(['channel_class', 'account_type', 'sales_basis', 'account_channel',
       'account_name', 'store_name', 'store_code', 'city', 'province',
       'region', 'bpc_region', 'sales_group', 'bpc_store_code', 'key',
       'plantilla_code', 'merchandiser', 'diser_coding', 'tos',
       '2025 Average Sales', 'January', 'February', 'March', 'April', 'May',
       'June', 'July', 'August', 'September', 'October', 'November',
       'December', 'January Target', 'February Target', 'March Target',
       'April Target', 'May Target', 'June Target', 'July Target',
       'August Target', 'September Target', 'October Target',
       'November Target', 'December Target'],
      dtype='object')


,channel_class,account_type,sales_basis,account_channel,account_name,store_name,store_code,city,province,region,bpc_region,key,plantilla_code,diser_coding,merchandiser,tos,sales_group,bpc_store_code,2025 Average Sales,January,January Target,February,February Target,March,March Target,April,April Target,May,May Target,June,June Target,July,July Target,August,August Target,September,September Target,October,October Target,November,November Target,December,December Target
0,Local Account,Outright,Sell In,Convenience Store,AANJ Prosperity and Wealth Mdse Corp.,Aanj Prosperity And Wealth Mdse Corp.,AANJ01,Valenzuela,Metro Manila,NCR,NCR,AANJ Prosperity and Wealth Mdse Corp.AANJ01,RIII-MPA-01,MPA,"Espino, Denn Mark",Angelita Illustre,S4,BPC-APW-00001,NaN,0.0,0,0.0,0,8629.18504,0,0.0,0,0.00000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
1,Local Account,Outright,Sell In,Others,"AB Leisure Exponent, Inc.","Ab Leisure Exponent, Inc.",ABLE01,Mandaluyong,Metro Manila,NCR,NCR,"AB Leisure Exponent, Inc.ABLE01",None,NaN,NaN,None,S4,BPC-ALE-00001,NaN,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0
2,Local Account,Outright,Sell In,Distributor,Abecure Wellness Products Trading,Abecure Wellness Products Trading,AWPT01,Iloilo City,Iloilo,Region VI,Visayas,Abecure Wellness Products TradingAWPT01,None,NaN,NaN,None,S4,BPC-AWP-00001,NaN,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0
3,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation Virac Sunrise,ACC01,Virac,Catanduanes,Region V,South Luzon,ACC Hypermart CorporationACC01,None,NaN,NaN,None,S4,BPC-AHC-00001,1169.55132,0.0,0,0.0,0,0.00000,0,0.0,0,5923.79424,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
4,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation,ACC02,Virac,Catanduanes,Region V,South Luzon,ACC Hypermart CorporationACC02,None,NaN,NaN,None,S4,BPC-AHC-00002,1567.79952,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0


In [169]:
# Join the S4 Targets and replace existing target values for S4 accounts
# sales_target_merge["stripped_store_code"] = sales_target_merge["store_code"].str.lstrip("0")

sales_S4_target_merge = pd.merge(sales_target_merge, S4_targets_2026, how="left", on="bpc_store_code")

for month in ["January","February","March","April","May","June","July",
              "August","September","October","November","December"]:
    s4_col = f"{month} S4 Target"
    target_col = f"{month} Target"
    mask = sales_S4_target_merge[s4_col].notna()
    sales_S4_target_merge.loc[mask, target_col] = sales_S4_target_merge.loc[mask, s4_col]

sales_S4_target_merge = sales_S4_target_merge.drop(columns=[f"{m} S4 Target" for m in
    ["January","February","March","April","May","June","July","August","September","October","November","December"]])

# sales_S4_target_merge = sales_S4_target_merge.drop(columns=["Account", "Store Code"])
display(sales_S4_target_merge.head())



,channel_class,account_type,sales_basis,account_channel,account_name,store_name,store_code,city,province,region,bpc_region,key,plantilla_code,diser_coding,merchandiser,tos,sales_group,bpc_store_code,2025 Average Sales,January,January Target,February,February Target,March,March Target,April,April Target,May,May Target,June,June Target,July,July Target,August,August Target,September,September Target,October,October Target,November,November Target,December,December Target
0,Local Account,Outright,Sell In,Convenience Store,AANJ Prosperity and Wealth Mdse Corp.,Aanj Prosperity And Wealth Mdse Corp.,AANJ01,Valenzuela,Metro Manila,NCR,NCR,AANJ Prosperity and Wealth Mdse Corp.AANJ01,RIII-MPA-01,MPA,"Espino, Denn Mark",Angelita Illustre,S4,BPC-APW-00001,NaN,0.0,0,0.0,0,8629.18504,0,0.0,0,0.00000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
1,Local Account,Outright,Sell In,Others,"AB Leisure Exponent, Inc.","Ab Leisure Exponent, Inc.",ABLE01,Mandaluyong,Metro Manila,NCR,NCR,"AB Leisure Exponent, Inc.ABLE01",None,NaN,NaN,None,S4,BPC-ALE-00001,NaN,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0
2,Local Account,Outright,Sell In,Distributor,Abecure Wellness Products Trading,Abecure Wellness Products Trading,AWPT01,Iloilo City,Iloilo,Region VI,Visayas,Abecure Wellness Products TradingAWPT01,None,NaN,NaN,None,S4,BPC-AWP-00001,NaN,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0
3,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation Virac Sunrise,ACC01,Virac,Catanduanes,Region V,South Luzon,ACC Hypermart CorporationACC01,None,NaN,NaN,None,S4,BPC-AHC-00001,1169.55132,0.0,3325.278658,0.0,2431.356026,0.00000,2765.818511,0.0,2008.250868,5923.79424,1728.789384,0.0,2081.987571,0.0,1684.882262,0.0,4780.717731,0.0,1948.023274,0.0,3444.610423,0.0,1980.790645,0.0,451.659048
4,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation,ACC02,Virac,Catanduanes,Region V,South Luzon,ACC Hypermart CorporationACC02,None,NaN,NaN,None,S4,BPC-AHC-00002,1567.79952,NaN,3325.278658,NaN,2431.356026,NaN,2765.818511,NaN,2008.250868,NaN,1728.789384,NaN,2081.987571,NaN,1684.882262,NaN,4780.717731,NaN,1948.023274,NaN,3444.610423,NaN,1980.790645,NaN,451.659048


In [170]:
# Include Hardcoded Alfamart Targets
target_columns = [
    "January Target", "February Target", "March Target",
    "April Target", "May Target", "June Target",
    "July Target", "August Target", "September Target",
    "October Target", "November Target", "December Target"
    ]

total_targets = {
    "January Target": 1016195,
    "February Target": 1529337,
    "March Target": 573009,
    "April Target": 5371457,
    "May Target": 750662,
    "June Target": 1236071,
    "July Target": 2284843,
    "August Target": 600937,
    "September Target": 600937,
    "October Target": 600937,
    "November Target": 600937,
    "December Target": 600937
}

mask = sales_S4_target_merge["account_name"] == "Alfamart Trading Philippines, Inc."
per_store_targets = pd.Series(total_targets) / 2305
sales_S4_target_merge.loc[mask, target_columns] = per_store_targets[target_columns].values

# display(sales_S4_target_merge[sales_S4_target_merge["account_name"] == "Alfamart Trading Philippines, Inc."].head(10))
# display(sales_S4_target_merge.loc[mask, target_columns].head())
# sales_S4_target_merge[target_columns].dtypes

In [171]:
# Write the final universe output file
final_universe_file = sales_S4_target_merge.copy()

# Fill the na rows with zeroes
final_universe_file.iloc[:, 18:] = final_universe_file.iloc[:, 18:].replace(r'^\s*$', 0, regex=True).fillna(0)
final_universe_file = final_universe_file.rename_axis("#").reset_index()
final_universe_file.head()

,#,channel_class,account_type,sales_basis,account_channel,account_name,store_name,store_code,city,province,region,bpc_region,key,plantilla_code,diser_coding,merchandiser,tos,sales_group,bpc_store_code,2025 Average Sales,January,January Target,February,February Target,March,March Target,April,April Target,May,May Target,June,June Target,July,July Target,August,August Target,September,September Target,October,October Target,November,November Target,December,December Target
0,0,Local Account,Outright,Sell In,Convenience Store,AANJ Prosperity and Wealth Mdse Corp.,Aanj Prosperity And Wealth Mdse Corp.,AANJ01,Valenzuela,Metro Manila,NCR,NCR,AANJ Prosperity and Wealth Mdse Corp.AANJ01,RIII-MPA-01,MPA,"Espino, Denn Mark",Angelita Illustre,S4,BPC-APW-00001,0.00000,0.0,0,0.0,0,8629.18504,0,0.0,0,0.00000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
1,1,Local Account,Outright,Sell In,Others,"AB Leisure Exponent, Inc.","Ab Leisure Exponent, Inc.",ABLE01,Mandaluyong,Metro Manila,NCR,NCR,"AB Leisure Exponent, Inc.ABLE01",None,NaN,NaN,None,S4,BPC-ALE-00001,0.00000,0.0,0,0.0,0,0.00000,0,0.0,0,0.00000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
2,2,Local Account,Outright,Sell In,Distributor,Abecure Wellness Products Trading,Abecure Wellness Products Trading,AWPT01,Iloilo City,Iloilo,Region VI,Visayas,Abecure Wellness Products TradingAWPT01,None,NaN,NaN,None,S4,BPC-AWP-00001,0.00000,0.0,0,0.0,0,0.00000,0,0.0,0,0.00000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
3,3,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation Virac Sunrise,ACC01,Virac,Catanduanes,Region V,South Luzon,ACC Hypermart CorporationACC01,None,NaN,NaN,None,S4,BPC-AHC-00001,1169.55132,0.0,3325.278658,0.0,2431.356026,0.00000,2765.818511,0.0,2008.250868,5923.79424,1728.789384,0.0,2081.987571,0.0,1684.882262,0.0,4780.717731,0.0,1948.023274,0.0,3444.610423,0.0,1980.790645,0.0,451.659048
4,4,Local Account,Outright,Sell In,Supermarket,ACC Hypermart Corporation,Acc Hypermart Corporation,ACC02,Virac,Catanduanes,Region V,South Luzon,ACC Hypermart CorporationACC02,None,NaN,NaN,None,S4,BPC-AHC-00002,1567.79952,0.0,3325.278658,0.0,2431.356026,0.00000,2765.818511,0.0,2008.250868,0.00000,1728.789384,0.0,2081.987571,0.0,1684.882262,0.0,4780.717731,0.0,1948.023274,0.0,3444.610423,0.0,1980.790645,0.0,451.659048


In [173]:
# Create universe excel file

date_today = datetime.now()

# Subtract 1 month
previous_month = date_today - relativedelta(months=1)
year_and_month = previous_month.strftime(r"%B %Y")

print("File name: ", year_and_month)
final_universe_file.to_excel(rf"V:\MIS Data\Universe\Python Generated Universe\Universe as of {year_and_month}.xlsx", index=False)

File name:  June 2026


In [174]:
# Universe Checker File

with pd.ExcelWriter(r"V:\MIS Data\Universe\Python Generated Universe\Universe Checker.xlsx") as writer:
    tos_names_merge.to_excel(writer, sheet_name="store_and_account_details", index=False)
    store_sales_2026.to_excel(writer, sheet_name="2026_store_sales", index=False)
    store_sales_2025.to_excel(writer, sheet_name="2025_store_sales", index=False)

# Columns to be added

# Week
# Day
# Frequency